# 06 - Capital Assembly

This notebook reviews the desk-level models-based capital assembly for the committed `capital_run_v1` fixture. It keeps NMRF mechanics out of scope: the SES total is loaded from the fixture's `expected_outputs.json` golden values so that notebook 04 can own method selection, valuation specs, reconciliation, and SES extraction.

Regulatory anchors: Basel MAR33 models-based capital concepts; U.S. NPR 2.0 proposed-rule parameters for IMCC, SES, supervisory multiplier, PLA add-on, and spot-versus-average binding term logic cited below; EU CRR Article 325ba as comparison context.

Prototype caution: this is a deterministic synthetic fixture walkthrough, not final regulatory capital or a supervisory capital report.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import load_capital_run_v1_fixture, policy_from_fixture
from frtb_ima.capital import models_based_capital
from frtb_ima.data_contracts import ScenarioCube
from frtb_ima.data_models import ModellabilityStatus
from frtb_ima.imcc import imcc_breakdown_for_policy
from frtb_ima.lha_builder import (
    nested_lh_vectors_from_cube,
    per_risk_class_nested_lh_vectors_from_cube,
)
from frtb_ima.nmrf import route_nmrf_classifications_for_capital

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


def filtered_cube(cube: ScenarioCube, risk_factor_names: tuple[str, ...]) -> ScenarioCube:
    allowed = set(risk_factor_names)
    selected = tuple(name for name in cube.risk_factor_names if name in allowed)
    if not selected:
        raise ValueError(f"no requested risk factors found in ScenarioCube '{cube.name}'")
    indices = [cube.risk_factor_index[name] for name in selected]
    values = cube.values[:, :, indices].copy()
    values.flags.writeable = False
    return ScenarioCube(
        values=values,
        scenario_metadata=cube.scenario_metadata,
        position_ids=cube.position_ids,
        risk_factor_names=selected,
        name=f"{cube.name}_imcc_population",
    )


def golden_value(name: str) -> float:
    return float(fixture.expected_outputs["scalars"][name]["value"])


def golden_check(name: str, actual: float) -> tuple[float, float, str]:
    spec = fixture.expected_outputs["scalars"][name]
    expected = float(spec["value"])
    tolerance = float(spec.get("atol", 1e-9)) + float(spec.get("rtol", 1e-9)) * abs(expected)
    diff = actual - expected
    return expected, diff, "PASS" if abs(diff) <= tolerance else "CHECK"


classifications = {
    name: ModellabilityStatus(status)
    for name, status in fixture.expected_outputs["classifications"].items()
}
routing = route_nmrf_classifications_for_capital(classifications, policy)
risk_factor_by_name = {risk_factor.name: risk_factor for risk_factor in fixture.risk_factors}
imcc_cube = filtered_cube(fixture.scenario_cube, routing.imcc_risk_factors)
imcc_risk_factors = tuple(risk_factor_by_name[name] for name in imcc_cube.risk_factor_names)
all_risk_class_vectors = nested_lh_vectors_from_cube(
    imcc_cube,
    imcc_risk_factors,
    lha_weights=policy.lha_weights,
)
per_risk_class_vectors = per_risk_class_nested_lh_vectors_from_cube(
    imcc_cube,
    imcc_risk_factors,
    lha_weights=policy.lha_weights,
)
imcc_result = imcc_breakdown_for_policy(
    all_risk_class_vectors,
    per_risk_class_vectors,
    policy,
    run_id=str(fixture.params["run_id"]),
    desk_id=str(fixture.params["desk_id"]),
)

ses_total = golden_value("total_ses")
multiplier = golden_value("supervisory_multiplier")
capital = models_based_capital(
    imcc_t_minus_1=imcc_result.imcc,
    ses_t_minus_1=ses_total,
    imcc_60d_avg=imcc_result.imcc * 1.03,
    ses_60d_avg=ses_total * 1.02,
    multiplier=multiplier,
    pla_addon=0.0,
)

spot_term = capital.imcc_t_minus_1 + capital.ses_t_minus_1
average_term = capital.multiplier * capital.imcc_60d_avg + capital.ses_60d_avg

display(
    Markdown(
        f"Loaded `{fixture.root.name}` for `{policy.regime.value}`. "
        f"Capital binding term: `{capital.binding_term}`. "
        "SES is loaded from committed expected outputs in this notebook."
    )
)


## Assembly Inputs

The capital assembly layer combines IMCC, SES, supervisory multiplier, and any PLA add-on. To avoid duplicating notebook 04, this notebook treats the committed SES total as an upstream reconciled input and focuses on the final assembly formula and binding term.


In [ ]:
source_rows = [
    ["IMCC t-1", "Recomputed here from fixture scenario cube via nested LH vectors", f"{capital.imcc_t_minus_1:,.6f}"],
    ["SES t-1", "Loaded from expected_outputs.json; NMRF chain is notebook 04", f"{capital.ses_t_minus_1:,.6f}"],
    ["IMCC 60d average", "Fixture deterministic average convention: IMCC t-1 x 1.03", f"{capital.imcc_60d_avg:,.6f}"],
    ["SES 60d average", "Fixture deterministic average convention: SES t-1 x 1.02", f"{capital.ses_60d_avg:,.6f}"],
    ["Multiplier", "Loaded from expected_outputs.json backtesting result", f"{capital.multiplier:.2f}"],
    ["PLA add-on", "Zero in this fixture because PLA zone is GREEN", f"{capital.pla_addon:,.6f}"],
]
display(markdown_table(["Input", "Fixture source", "Value"], source_rows))


## Binding Term Logic

The prototype formula compares the spot term with the supervisory-multiplied 60-business-day average term, then adds the PLA add-on. The `CapitalComponents.binding_term` field makes the selected side explicit for audit review.


In [ ]:
binding_rows = [
    [
        "SPOT",
        "IMCC t-1 + SES t-1",
        f"{capital.imcc_t_minus_1:,.2f} + {capital.ses_t_minus_1:,.2f}",
        f"{spot_term:,.2f}",
        "YES" if capital.binding_term == "SPOT" else "NO",
    ],
    [
        "AVERAGE",
        "multiplier x IMCC 60d avg + SES 60d avg",
        f"{capital.multiplier:.2f} x {capital.imcc_60d_avg:,.2f} + {capital.ses_60d_avg:,.2f}",
        f"{average_term:,.2f}",
        "YES" if capital.binding_term == "AVERAGE" else "NO",
    ],
    ["PLA add-on", "Added after binding term selection", "GREEN fixture", f"{capital.pla_addon:,.2f}", "n/a"],
    ["Models-based capital", "max(SPOT, AVERAGE) + PLA add-on", "", f"{capital.models_based_capital:,.2f}", capital.binding_term],
]
display(markdown_table(["Term", "Formula", "Inputs", "Value", "Binding"], binding_rows))


## Golden Output Reconciliation

The notebook should make fixture expectations visible rather than silently re-deriving them. The rows below compare the recomputed or loaded assembly values against `expected_outputs.json`.


In [ ]:
reconciliation_rows = []
for label, name, actual in [
    ("IMCC", "imcc", imcc_result.imcc),
    ("Total SES", "total_ses", ses_total),
    ("Supervisory multiplier", "supervisory_multiplier", multiplier),
    ("Models-based capital", "models_based_capital", capital.models_based_capital),
]:
    expected, diff, status = golden_check(name, actual)
    reconciliation_rows.append(
        [label, f"{actual:,.6f}", f"{expected:,.6f}", f"{diff:,.3e}", status]
    )

reconciliation_rows.append(
    [
        "Binding term",
        capital.binding_term,
        str(fixture.expected_outputs["capital"]["binding_term"]),
        "",
        "PASS" if capital.binding_term == fixture.expected_outputs["capital"]["binding_term"] else "CHECK",
    ]
)

display(
    markdown_table(
        ["Output", "Notebook", "Expected output", "Delta", "Status"],
        reconciliation_rows,
    )
)


## Capital Assembly Visual

The stacked bars show the two candidate terms before the PLA add-on. The horizontal line is the final models-based capital value for the fixture.


In [ ]:
labels = ["SPOT", "AVERAGE"]
imcc_terms = np.asarray([capital.imcc_t_minus_1, capital.multiplier * capital.imcc_60d_avg])
ses_terms = np.asarray([capital.ses_t_minus_1, capital.ses_60d_avg])
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.bar(x, imcc_terms, color="#4c78a8", label="IMCC term")
ax.bar(x, ses_terms, bottom=imcc_terms, color="#f28e2b", label="SES term")
ax.axhline(
    capital.models_based_capital,
    color="#c44e52",
    linewidth=1.8,
    label="Models-based capital",
)
for index, total in enumerate(imcc_terms + ses_terms):
    marker = "binding" if labels[index] == capital.binding_term else "candidate"
    ax.text(index, total, f"{total:,.0f}\n{marker}", ha="center", va="bottom")
ax.set_xticks(x, labels)
ax.set_ylabel("Capital measure")
ax.set_title("Spot versus average term before PLA add-on")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
